# Pre-training mini-GPT on Kaggle

Required notebook settings: accelerator **GPU T4**, **Internet on**, and the tokenized dataset attached (it must contain `train.bin`, `val.bin` and `tokenizer.json`).

Run it with **Save & Run All (Commit)** so training continues in the background.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/mini-gpt.git"
REPO_DIR = "/kaggle/working/mini-gpt"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

In [ ]:
DATA_DIR = "/kaggle/input/minigpt-tinystories"
RUN_DIR = "/kaggle/working/runs/pretrain"
RESUME_FROM = None

if RESUME_FROM:
    import shutil

    os.makedirs(RUN_DIR, exist_ok=True)
    for name in ("last.pt", "best.pt", "metrics.jsonl"):
        source = os.path.join(RESUME_FROM, name)
        if os.path.exists(source):
            shutil.copy(source, RUN_DIR)

print(os.listdir(DATA_DIR))

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
!nvidia-smi

In [ ]:
!python -m minigpt.training.pretrain --resume --override data.train={DATA_DIR}/train.bin data.val={DATA_DIR}/val.bin train.out_dir={RUN_DIR}

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{RUN_DIR}/metrics.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]
train = [r for r in records if r["type"] == "train"]
evals = [r for r in records if r["type"] == "eval"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train], [r["loss"] for r in train], label="train")
axes[0].plot([r["step"] for r in evals], [r["val_loss"] for r in evals], label="validation")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[1].plot([r["step"] for r in train], [r["tokens_per_sec"] for r in train])
axes[1].set_xlabel("step")
axes[1].set_ylabel("tokens / s")
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/curves.png", dpi=150)
plt.show()